# 06 · clustering and label transfer

We have a Harmonized lipid embedding, two coronal sections of the same brain plane (one control, one
pregnant), and no group labels yet. This notebook turns that embedding into **lipid territories** and
gives each pixel a name. The path has three movements.

First we **cluster**: we build a graph of who-neighbours-whom in the lipid embedding and let a
community-finder cut the pixels into groups. Those groups are the **lipizones**, the data-driven
territories of lipid composition that the Lipid Brain Atlas is built on.

Second we **compare to anatomy**: we color the brain section by lipizone, lay it next to the Allen
reference, and measure with one number (the Adjusted Rand Index) how much the lipid territories agree
with classical anatomy. The answer is the heart of the lesson: they overlap, but lipizones are their
own thing.

Third we **transfer labels**: we learn the lipizones on the control brain only, then copy them onto the
pregnant brain by asking each pregnant pixel which control pixels it sits next to in the shared space.
That is exactly how the Atlas annotates new brains from few sections, and it is what lets us put control
and pregnant side by side speaking the same vocabulary.

We close with the production path: a cell that git-clones EUCLID and calls its real clustering, framed
so you see that the package does a more careful, divisive version of the same idea you just built by
hand. You will build that divisive idea from scratch in the next notebook.

The callout markers are the same as everywhere in the course:

- 🔬 **TASK** something for you to write or run.
- 💡 **HINT** a nudge when a task is non-obvious.
- ❓ **QUESTION** a thinking prompt, no code required.
- ⚠️ **CHECKPOINT** what you should see if the cell worked; if you do not see it, stop and debug.


In [ ]:
# the stack we use today, plus the course helpers and the lab figure style
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import KNeighborsClassifier

import cajal_lipidomics as cl
from cajal_lipidomics import embedding as emb, analysis as an, plotting as pl
from cajal_lipidomics.style import set_style, FS
set_style()

# one seed for everything random today, so the notebook is reproducible
RNG_SEED = 0
np.random.seed(RNG_SEED)

print("ready: numpy", np.__version__, "| scanpy", sc.__version__)

⚠️ **CHECKPOINT.** You should see `ready: numpy ... | scanpy ...` and no red error. `set_style()`
applies the lab defaults so every figure comes out clean and Illustrator-editable without you fussing
over it.

## 1 · load the two sections and pull out the embedding

We load the section pair. It is one AnnData object holding **174768 pixels** stacked from two coronal
sections of the *same* plane of the mouse brain (anterior-posterior level around 6.5). One section is a
control female (`Condition == "naive"`, `SectionID 75`), the other is pregnant at embryonic day 13.5
(`Condition == "pregnant"`, `SectionID 110`). Each pixel is one MALDI desorption point, a tiny laser
spot of tissue, not one cell: it mixes cell bodies, axons, dendrites, and glial projections.

`adata.X` is the **uMAIA-normalized** lipid matrix, **173 lipids** across the pixels. `adata.obs`
carries the per-pixel metadata: which section and condition, the Allen Common Coordinate Framework
coordinates `zccf`/`yccf`, the Allen anatomical `acronym` of the region each pixel falls in, and the
published `lipizone_names` we will use as a sanity reference.

The crucial extra ingredient lives in `adata.obs` too: **16 columns named `globalembeddings_0` through
`globalembeddings_15`**. These are the Harmonized NMF factors, the per-pixel embedding the previous
notebook built. NMF compressed the 100-odd lipids into 16 additive, non-negative **lipid programs**;
Harmony then removed the section-to-section offset so the two sections live in one comparable space.
That 16-number-per-pixel matrix is the input to everything today. We never cluster on the raw 173
lipids directly, we cluster on these 16 summary coordinates.

In [ ]:
# load the substrate: 174768 pixels x 173 lipids, two sections of the same plane
adata = ad.read_h5ad("../../data/sections_pair.h5ad")
print("pixels x lipids:", adata.shape)

# pull the 16 Harmonized NMF factors out of obs into one (pixels x 16) array
emb_cols = [c for c in adata.obs.columns if c.startswith("globalembeddings_")]
G = adata.obs[emb_cols].to_numpy().astype(float)
print("embedding shape:", G.shape, "| factors:", len(emb_cols))

# boolean masks for the two halves of the data
is_control = (adata.obs["Condition"] == "naive").to_numpy()
is_pregnant = (adata.obs["Condition"] == "pregnant").to_numpy()
print(f"control pixels: {is_control.sum():>6}   pregnant pixels: {is_pregnant.sum():>6}")

⚠️ **CHECKPOINT.** `pixels x lipids: (174768, 173)`, `embedding shape: (174768, 16)`, and roughly
84321 control plus 90447 pregnant pixels. The 16-factor matrix `G` is the lipid embedding we cluster
on; the masks split it into the reference (control) and the query (pregnant) we will transfer onto.

Before any clustering, look at the embedding. Each of the 16 factors is a lipid program, and its
value at a pixel says how strongly that program is expressed there. Painting a few factors back onto the
brain coordinates is the fastest way to see that these numbers are not random: they carry sharp
anatomical structure, which is exactly what makes them clusterable.

In [ ]:
# paint the first four NMF factors onto the control section's CCF coordinates
obs = adata.obs
z = obs["zccf"].to_numpy(); y = -obs["yccf"].to_numpy()      # CCF coords; flip y so dorsal is up
m = is_control                                               # show one section for clarity

fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
for k, ax in enumerate(axes):
    v = G[:, k]
    sc_ = ax.scatter(z[m], y[m], c=v[m], cmap="plasma", s=3, rasterized=True,
                     vmin=np.quantile(v[m], 0.02), vmax=np.quantile(v[m], 0.98))
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"NMF factor {k}", fontsize=FS["m"])
    for s in ax.spines.values(): s.set_visible(False)
fig.suptitle("four lipid programs painted on the control section", fontsize=FS["l"])
plt.tight_layout(); plt.show()

⚠️ **CHECKPOINT.** Each panel shows crisp territories, not noise: one factor lights up the
fiber tracts, another the cortical ribbon, another a deep nucleus. The factors already *are* a coarse
map of the brain. Clustering will sharpen these continuous programs into discrete, named regions.

❓ **QUESTION.** Why cluster on the 16 factors instead of the 173 raw lipids? Two reasons hide in the
plot above. The factors are already de-noised and de-batched (Harmony ran on them), and 16 numbers make
the neighbour graph far cheaper to build than 173. Clustering on the embedding is clustering on the
*signal*, with the section artifact and the redundancy already removed.

## 2 · the kNN graph: write down who-is-near-whom

We want to carve the pixels into groups without telling the algorithm how many groups to expect. The
first step is to turn the cloud of 16-dimensional points into a **graph**, a network of connections.

**The idea, in one sentence.** For each pixel, find its **k nearest neighbours** in the embedding, the k
other pixels whose 16 factors are most similar, and draw an edge to each of them. Do this for every
pixel and you get a network where pixels with similar lipid composition are wired together and pixels
with different composition are not. "Distance" here is ordinary Euclidean distance in the 16-dimensional
factor space, the straight-line distance you would compute with Pythagoras, just in 16 dimensions
instead of 2.

That graph is the entire input to community detection. It throws away the exact coordinates and keeps
only the relationships: who is close to whom. Two pixels on opposite sides of the brain that happen to
have the same lipid mix end up neighbours in the graph even though they are far apart physically, which
is the point: we are grouping by lipid identity, not by location.

Let us build it transparently on a small random sample first, so you can see the graph is just a matrix
of edges, before we hand the full job to scanpy.

In [ ]:
# build a kNN graph BY HAND on a small sample, to see there is no magic in it
from sklearn.neighbors import NearestNeighbors

rng = np.random.default_rng(RNG_SEED)
sample = rng.choice(np.where(is_control)[0], size=2000, replace=False)   # 2000 control pixels
Gs = G[sample]

k = 15                                                  # 15 nearest neighbours per pixel
nn = NearestNeighbors(n_neighbors=k + 1).fit(Gs)        # +1 because the closest point is itself
dist, idx = nn.kneighbors(Gs)
dist, idx = dist[:, 1:], idx[:, 1:]                     # drop the self-match in column 0

print("each of", Gs.shape[0], "pixels is linked to its", k, "nearest neighbours")
print("so the graph has", Gs.shape[0] * k, "directed edges")
print("example: pixel 0's neighbours are rows", idx[0][:5], "...")
print("their distances in factor space:", np.round(dist[0][:5], 5))

⚠️ **CHECKPOINT.** You built a graph: 2000 pixels, each linked to its 15 nearest, so 30000
edges. `idx[0]` lists the row numbers of pixel 0's neighbours and `dist[0]` their distances. That edge
list *is* the graph. Nothing more mysterious than a table of "who is near whom".

A picture makes it concrete. We lay the 2000 sampled pixels out by two of their NMF factors and
draw a line for every edge. Dense thickets of lines are pixels that all resemble each other, the future
clusters; sparse regions are the gaps between groups.

In [ ]:
# draw the kNN graph: points placed by two factors, a faint line per edge
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for i in range(Gs.shape[0]):                            # one short segment per neighbour
    for j in idx[i]:
        ax.plot([Gs[i, 0], Gs[j, 0]], [Gs[i, 1], Gs[j, 1]],
                lw=0.1, color="0.6", alpha=0.3, rasterized=True)
ax.scatter(Gs[:, 0], Gs[:, 1], s=6, c="crimson", rasterized=True, zorder=3)
ax.set_xlabel("NMF factor 0"); ax.set_ylabel("NMF factor 1")
ax.set_title("kNN graph on 2000 control pixels (k=15)", fontsize=FS["m"])
plt.tight_layout(); plt.show()

⚠️ **CHECKPOINT.** You see clumps of densely connected red points joined by a haze of thin grey
lines, with thinner bridges between clumps. The community-finder we run next is just an algorithm that
spots those clumps automatically.

💡 **HINT.** We plotted only factors 0 and 1, so two pixels can *look* far apart on screen yet be close
neighbours through the other 14 dimensions. The graph is honest about all 16; the scatter is a flattened
shadow of it. That is also why a 2D plot can never fully justify a clustering: trust the graph, use the
picture as intuition.

## 3 · Leiden: cut the graph into communities

Now the graph gets cut into groups. **Leiden** is a community-detection algorithm: it scans the graph
and finds sets of pixels that are densely wired to each other and only sparsely wired to the rest,
exactly how you would spot friend-groups in a social network. Each community becomes one cluster. It
never sees coordinates or anatomy, only the edges, and it decides the number of clusters on its own.

The score Leiden maximizes is **modularity**: roughly, the fraction of edges that fall *inside*
communities minus what you would expect if the same number of edges were thrown down at random. A
grouping where almost every edge stays within a group scores high; a grouping that cuts through dense
thickets scores low. Leiden shuffles pixels between communities, greedily, until modularity stops
improving, then refines.

Two knobs control the outcome:

- **`n_neighbors`** (the k from section 2): how local the graph is. Small k builds a fine, fragmented
  graph; large k builds a smoother, broader one.
- **`resolution`**: how eagerly Leiden splits. The number multiplies the penalty for leaving edges
  *between* communities, so a higher resolution makes the algorithm prefer many small tight groups, a
  lower resolution prefers a few big loose ones. This is the dial you turn to set the granularity of
  your lipizones.

We learn the clusters on the **control section only**. That is deliberate: control is our reference
brain, and in section 5 we will transfer its labels onto the pregnant brain rather than re-cluster it.
scanpy wraps the kNN graph and Leiden in two calls, and they are the exact functions the Atlas pipeline
uses.

🔬 **TASK.** Wrap the control embedding in an AnnData, build the neighbour graph with
`sc.pp.neighbors`, and run `sc.tl.leiden`. Use `n_neighbors=15`, `resolution=1.0`, and the seed so it is
reproducible. Print how many clusters came out and how big they are.

In [ ]:
# cluster the CONTROL pixels on their 16-factor embedding, via scanpy
# 🔬 your code here


⚠️ **CHECKPOINT.** Leiden returns on the order of two dozen clusters (24 with this seed and
resolution), ranging from a few hundred to several thousand pixels each. These are your candidate
lipizones, found with zero supervision: no anatomy, no condition, just the lipid embedding graph.

💡 **HINT.** The helper `cl.embedding.leiden_clusters(G[is_control], n_neighbors=15, resolution=1.0)`
does exactly the three lines above (wrap, neighbours, leiden) and hands back the label array. We unrolled
it so the kNN-graph-then-community-cut is not a black box; from here you can call the helper.

In [ ]:
# the helper packs the same recipe; confirm it gives an equivalent clustering
leiden_helper = emb.leiden_clusters(G[is_control], n_neighbors=15, resolution=1.0, random_state=RNG_SEED)
print("helper clusters:", len(np.unique(leiden_helper)),
      "| same count as hand-rolled:", len(np.unique(leiden_helper)) == n_clusters)

Let us turn the resolution dial once, to feel it. We re-run Leiden lower and higher and just count
clusters. Nothing about the data changed; only how finely we asked it to be carved.

In [ ]:
# sweep resolution to see it controls granularity (this is the main lipizone dial)
for res in [0.3, 1.0, 3.0]:
    lab = emb.leiden_clusters(G[is_control], n_neighbors=15, resolution=res, random_state=RNG_SEED)
    print(f"resolution {res:>4}  ->  {len(np.unique(lab)):>3} clusters")

⚠️ **CHECKPOINT.** Cluster count climbs monotonically with resolution: a handful at 0.3, a couple
dozen at 1.0, many more at 3.0. The real Atlas uses a very high effective resolution (the paper compares
to Leiden at resolution 14) to reach hundreds of fine lipizones. There is no single correct number; the
right granularity is a scientific judgement about how fine a territory is still biologically meaningful.

## 4 · the lipid territories in space, and how they track anatomy

Numbers on a graph mean nothing until we put them back on the brain. We color every control pixel by its
Leiden cluster on the CCF coordinates. If the clustering captured real biology, the clusters should be
**spatially coherent**: contiguous patches, not salt-and-pepper static, because neighbouring tissue
tends to share lipid composition.

In [ ]:
# paint the Leiden clusters back onto the control section
zc, yc = z[is_control], y[is_control]
labels = leiden_ctrl.astype(int)
# a distinct color per cluster from a qualitative colormap
palette = plt.cm.tab20(np.linspace(0, 1, max(20, labels.max() + 1)))

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(zc, yc, c=palette[labels % len(palette)], s=4, rasterized=True)
ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
for s in ax.spines.values(): s.set_visible(False)
ax.set_title(f"Leiden lipizones on the control section ({n_clusters} clusters)", fontsize=FS["m"])
plt.tight_layout(); plt.show()

⚠️ **CHECKPOINT.** The clusters draw clean, contiguous territories: a ribbon hugging the cortical
surface, a bright wedge in the white-matter tracts, blocks filling the deep nuclei. They were found from
lipid composition alone, yet they carve the section into anatomy-like shapes. That spatial coherence is
the first sign the lipizones are real and not noise.

Now lay the lipid territories beside the **Allen reference**, the classical anatomy. Each pixel
already carries `allencolor`, the official color of the Allen region it falls in. Plotting our Leiden
clusters next to the Allen coloring is the eyeball test of how much the two partitions agree.

In [ ]:
# side by side: our Leiden lipizones vs the Allen anatomical regions, same section
fig, axes = plt.subplots(1, 2, figsize=(12, 5.6))
axes[0].scatter(zc, yc, c=palette[labels % len(palette)], s=4, rasterized=True)
axes[0].set_title("Leiden lipizones (from lipids)", fontsize=FS["m"])
axes[1].scatter(zc, yc, c=adata.obs["allencolor"].to_numpy()[is_control], s=4, rasterized=True)
axes[1].set_title("Allen regions (classical anatomy)", fontsize=FS["m"])
for ax in axes:
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)
plt.tight_layout(); plt.show()

The shapes rhyme but do not match. Some Allen regions split into several lipizones, several Allen
regions merge into one lipizone, and a few boundaries land in different places entirely. We need a single
number to say how much they agree, beyond the eyeball.

That number is the **Adjusted Rand Index (ARI)**. The plain Rand Index asks: take every possible pair of
pixels, and check whether the two clusterings agree on them, that is, whether both put the pair together
or both keep the pair apart. The fraction of pairs they agree on is the Rand Index. The *adjusted*
version subtracts the agreement you would expect from two random labelings of the same sizes and
rescales, so that **1.0 means identical partitions, 0.0 means no better than chance, and negative means
worse than chance**. It is the standard way to compare two clusterings when the cluster names are
arbitrary, which ours are: Leiden's "cluster 7" has no reason to equal Allen's "CP".

🔬 **TASK.** Compute the ARI between your Leiden labels and the Allen `acronym` on the control
pixels. Then compute it for the *published* lipizones (`lipizone_names`) against the same Allen acronyms,
so we have a reference for what a careful, finished lipizone partition scores.

In [ ]:
# ARI: how much do the lipid territories agree with classical anatomy?
# 🔬 your code here


⚠️ **CHECKPOINT.** Both ARIs are small and positive, around 0.05. Our quick Leiden scores about
0.05, the carefully built published lipizones score about 0.06: the same ballpark. A small positive ARI
means the two partitions **agree well above chance but are far from identical**.

❓ **QUESTION.** This is the central result of the notebook, so read the number carefully. An ARI near
0.05 is not a failure of the clustering. It is the biology. Lipizones are defined by **membrane lipid
composition**, and lipid composition is shaped by cell types, myelination, and metabolic state that do
*not* respect the cytoarchitectural boundaries Allen drew from cell density and Nissl stains. The two
maps overlap because the same tissue underlies both, but they disagree because they measure different
things. The published Atlas makes exactly this point: lipizones partially track anatomy, yet they are
their own organizing principle of the brain. If the ARI were near 1.0, lipidomics would just be
re-deriving anatomy and would tell us nothing new.

We can see *where* they agree and disagree with a confusion table. We cross-tabulate Leiden
clusters against the most common Allen regions and read which lipid territory maps onto which anatomical
region. A clean block structure (each Leiden cluster concentrated in one or two Allen regions) is partial
agreement; a smeared table is independence.

In [ ]:
# crosstab: Leiden cluster (rows) vs the 12 most common Allen regions (cols)
top_regions = pd.Series(allen_ctrl).value_counts().head(12).index
sub = pd.Series(allen_ctrl).isin(top_regions).to_numpy()
ct = pd.crosstab(pd.Series(leiden_ctrl[sub], name="Leiden"),
                 pd.Series(allen_ctrl[sub], name="Allen region"))
ct = ct[top_regions]                                       # order cols by frequency

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(np.log1p(ct.values), aspect="auto", cmap="magma")
ax.set_xticks(range(ct.shape[1])); ax.set_xticklabels(ct.columns, rotation=90, fontsize=FS["xs"])
ax.set_yticks(range(ct.shape[0])); ax.set_yticklabels(ct.index, fontsize=FS["xs"])
ax.set_xlabel("Allen region"); ax.set_ylabel("Leiden lipizone")
ax.set_title("where lipizones and anatomy overlap (log pixel counts)", fontsize=FS["m"])
cl.style.lightweight_colorbar(im, ax, label="log(1 + pixels)")
plt.tight_layout(); plt.show()

⚠️ **CHECKPOINT.** Each row tends to brighten in just one or two columns, not spread evenly: a
given lipid territory lives mostly inside one or two anatomical regions, which is the partial agreement
the ARI quantified. But a single Allen region (a bright column) often receives several lipizone rows,
the lipidome splitting one anatomical region into finer compositional sub-territories. That asymmetry is
the resolution of lipidomics beyond anatomy.

To name a lipid territory you ask which lipids define it. The helper `cl.analysis.marker_lipids`
runs a one-versus-rest comparison per cluster and ranks the lipids most elevated inside it. We attach the
Leiden labels and read the markers of the largest clusters.

In [ ]:
# marker lipids per Leiden cluster (one-vs-rest log2 fold change), control only
adata_ctrl = adata[is_control].copy()
adata_ctrl.obs["leiden"] = pd.Categorical(leiden_ctrl)
markers = an.marker_lipids(adata_ctrl, "leiden", top_n=4)

# show markers for the four largest clusters
biggest = pd.Series(leiden_ctrl).value_counts().head(4).index
for cluster_id in biggest:
    top = markers.get(str(cluster_id), markers.get(cluster_id, []))
    pretty = ", ".join(f"{name} (log2FC {lfc:+.2f})" for name, lfc in top)
    print(f"cluster {cluster_id:>2} ({(leiden_ctrl == str(cluster_id)).sum():>5} px):  {pretty}")

⚠️ **CHECKPOINT.** Each cluster prints its top lipids with a positive log2 fold change versus the
rest of the brain. The four largest territories here are headed by sphingomyelins such as SM 36:2;O2 and
particular phosphatidylcholines (PC 38:5, PC 32:0), with phosphoinositides (PI), phosphatidic acid (PA),
and ceramides (Cer) filling out their top-four; the white-matter HexCer (myelin) territories show up
among the smaller clusters, which is why no HexCer leads this top-four of the biggest groups. The markers
are how a nameless "cluster 7" becomes a biologically meaningful lipizone: it is defined by the lipids it
concentrates.

## 5 · label transfer: carry the control lipizones onto the pregnant brain

We clustered the control section. We do **not** want to cluster the pregnant section separately and then
puzzle out which of its new clusters corresponds to which control cluster. We want to *carry the existing
labels over* so both sections speak the same vocabulary and we can compare like with like.

**The idea.** Same kNN trick as Leiden, used differently. For each **query** pixel (a pregnant pixel,
unlabelled) find its nearest **reference** pixels (control pixels, already labelled) in the shared
embedding, and take a **majority vote** of their labels. The pixel inherits whatever its neighbours
mostly are. A bonus falls out for free: how *unanimous* the neighbours are gives a **confidence**. A
pregnant pixel whose 15 control neighbours all carry lipizone 4 is a confident call; one split eight-to-
seven is a flag to inspect.

**The catch, and why we need a shared space.** A kNN vote only makes sense if control and pregnant pixels
live in one comparable coordinate system. The raw NMF factors carry a section-to-section offset (a batch
effect): the pregnant section was acquired on a different day, so its absolute factor values drift even
where the biology is identical. If we voted in that drifted space, every pregnant pixel would just find
its nearest *pregnant* neighbours and the transfer would be meaningless. **Harmony** fixes this. It
removes the per-section offset from the embedding so that biologically similar pixels from the two
sections sit on top of each other, leaving a space where cross-section neighbours are honest.

One careful choice: we harmonize **only on `SectionID`**, not on `Condition`. With exactly two sections,
one per condition, section and condition are perfectly confounded; if we told Harmony to also erase the
condition difference it would scrub out the very pregnancy biology we want to study. Removing the section
offset is enough to make the spaces comparable while leaving the biology intact.

In [ ]:
# Harmony on the 16-factor embedding, batch = SectionID only (NOT Condition)
# this is for clustering / label transfer ONLY; the differential test uses the raw uMAIA data
Z = emb.harmonize(G, adata.obs["SectionID"].astype(str).to_numpy(), random_state=RNG_SEED)
print("Harmonized embedding:", Z.shape)

# split the shared space into reference (control, labelled) and query (pregnant, to label)
Z_ref,  y_ref  = Z[is_control], leiden_ctrl                 # control + its Leiden lipizones
Z_query        = Z[is_pregnant]                             # pregnant, no labels yet
print(f"reference: {Z_ref.shape[0]} labelled control pixels")
print(f"query    : {Z_query.shape[0]} pregnant pixels to annotate")

⚠️ **CHECKPOINT.** `Harmonized embedding: (174768, 16)`, then the split into about 84k labelled
control reference pixels and 90k pregnant query pixels. `Z` is the shared space; `y_ref` is the control
lipizone label for each reference pixel. Now we vote.

🔬 **TASK.** Do the transfer by hand with scikit-learn's `KNeighborsClassifier` so you see there
is no magic. Fit it on the labelled control reference, predict on the pregnant query, and read off the
neighbour-vote confidence as the max class probability per pixel.

In [ ]:
# kNN majority vote: fit on control, transfer onto pregnant
# 🔬 your code here


⚠️ **CHECKPOINT.** All 90447 pregnant pixels get a lipizone, with a mean confidence around 86%.
The confidence is genuinely informative: most pixels sit deep inside a territory and vote nearly
unanimously, while a minority near boundaries split their vote and come out lower. Those low-confidence
pixels are the ones a curator would review.

💡 **HINT.** `cl.embedding.knn_transfer(Z_ref, y_ref, Z_query, k=15)` packs exactly this fit-predict-
confidence into one call and returns `(pred, conf)`. We unrolled it so the majority vote is visible; the
helper does precisely the same thing.

In [ ]:
# the helper does the same vote in one line
pred_h, conf_h = emb.knn_transfer(Z_ref, y_ref, Z_query, k=15)
print("helper agrees with hand-rolled prediction on",
      f"{(pred_h == pred_pregnant).mean()*100:.1f}% of pregnant pixels")

Now the payoff: control and pregnant, side by side, in the **same lipizone vocabulary**. The
control panel shows the Leiden lipizones we learned; the pregnant panel shows those same labels
transferred by the kNN vote. Because the colors mean the same thing in both, any shift in a territory's
size or position between the two is a candidate biological change.

In [ ]:
# build a full-length label array (control = learned, pregnant = transferred), then map to colors
labels_all = np.empty(adata.n_obs, dtype=object)
labels_all[is_control]  = leiden_ctrl
labels_all[is_pregnant] = pred_pregnant
labels_all_int = labels_all.astype(int)

zp, yp = z[is_pregnant], y[is_pregnant]
fig, axes = plt.subplots(1, 2, figsize=(12, 5.6))
axes[0].scatter(zc, yc, c=palette[leiden_ctrl.astype(int) % len(palette)], s=4, rasterized=True)
axes[0].set_title("control: lipizones learned with Leiden", fontsize=FS["m"])
axes[1].scatter(zp, yp, c=palette[pred_pregnant.astype(int) % len(palette)], s=4, rasterized=True)
axes[1].set_title("pregnant: same lipizones, transferred by kNN vote", fontsize=FS["m"])
for ax in axes:
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)
plt.tight_layout(); plt.show()

⚠️ **CHECKPOINT.** The two sections show the same colored territories in the same anatomical
places: the cortical ribbon, the white-matter wedge, the deep nuclei all reappear in matching colors.
That correspondence is the whole point of label transfer: we annotated the pregnant brain without
clustering it, just by asking each pixel which control pixels it resembles. Now the two are directly
comparable.

Finally, map the transfer **confidence** onto the pregnant section. Bright pixels are confident
calls deep inside a territory; dim pixels are uncertain calls, and they should fall on the **boundaries**
between lipizones, where neighbours of two kinds compete for the vote.

In [ ]:
# paint transfer confidence on the pregnant section
fig, ax = plt.subplots(figsize=(6.8, 6))
scc = ax.scatter(zp, yp, c=conf_pregnant, cmap="viridis", s=4, rasterized=True, vmin=0.5, vmax=1.0)
ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
for s in ax.spines.values(): s.set_visible(False)
ax.set_title("label-transfer confidence on the pregnant section", fontsize=FS["m"])
cl.style.lightweight_colorbar(scc, ax, label="neighbour-vote confidence")
plt.tight_layout(); plt.show()

⚠️ **CHECKPOINT.** Territory interiors glow bright (high confidence) and thin dim lines trace
their borders, exactly where a pixel's lipid composition is intermediate and its control neighbours
disagree. The confidence map is essentially an automatic uncertainty map of the annotation, and it
points a curator straight at the pixels worth a second look.

❓ **QUESTION.** Both Leiden (section 3) and label transfer (this section) run the *same* nearest-
neighbour idea on the *same* Harmonized embedding. One groups pixels from scratch, the other copies
existing labels onto new pixels. Why does it make sense that both operate on the Harmonized embedding
rather than on the raw 173 lipids? Hint: think about what would happen to cross-section neighbours if the
section batch offset were still in the coordinates.

## 6 · the production path: EUCLID does a more careful version of this

What you built is the honest skeleton of how the Lipid Brain Atlas finds and transfers lipizones: a kNN
graph, a community cut, a kNN vote across a Harmonized space. The published Atlas uses a more
sophisticated engine called **EUCLID** (Enhanced uMAIA for Clustering Lipizones, Imputation, and
Differential analysis). It is worth understanding how it differs, because the next notebook asks you to
rebuild its core idea from scratch.

Where we ran **one flat Leiden** at a single resolution, EUCLID runs a **divisive, top-down binary
splitter**. It starts with all pixels in one group and recursively asks "should this group split in
two?". At each candidate split it recomputes a *local* NMF on just the pixels in that branch (so fine
structure that the global 16 factors blur out becomes visible), over-segments into 60 micro-clusters with
k-means, then re-aggregates those 60 into exactly 2 groups. A split is accepted only if it passes three
gates: the two halves differ in at least a couple of lipids (a Mann-Whitney test), each half is big
enough, and each half is spatially coherent across sections rather than salt-and-pepper. The result is a
tree, and the terminal leaves are the lipizones, organized into the class, subclass, supertype, lipizone
hierarchy the Atlas reports.

Where we transferred labels with a **kNN vote**, EUCLID trains an **XGBoost classifier at every node** of
that tree. To annotate a new pixel it walks the tree from the root, at each node applying the local NMF
and the trained classifier to decide left or right, until the pixel lands in a leaf. That makes the
transfer reproducible and as fine-grained as the tree, which is how the Atlas label-transferred 215 of
222 supertypes onto the pregnant brains.

The next cell shows the real production call. It git-clones EUCLID and invokes
`learn_euclid_clustering` on the control brain and `apply_euclid_clustering` on the full data, the two
functions at the heart of the package. This is your first encounter with cloning a repository and calling
a package you did not write. We mark the cell **illustrative**: the full divisive recursion takes hours
on this many pixels and pulls in heavy dependencies, so we do not execute it here. Read it as the shape
of the real thing; you have already run the transparent version of every step it performs.

In [ ]:
# ILLUSTRATIVE: the real EUCLID production path. Do NOT run in the base course env:
# the divisive recursion takes hours on 174768 pixels and needs the full EUCLID stack.
# It is shown so you see the shape of the package call you will rebuild from scratch next.

# 1) clone the package from GitHub (your first `git clone` of a real analysis package)
#    !git clone https://github.com/lamanno-epfl/EUCLID.git
#    %pip install -e ./EUCLID

# 2) import the two functions at the heart of EUCLID's clustering module
# from euclid.clustering import Clustering, learn_euclid_clustering, apply_euclid_clustering

# 3) LEARN the divisive lipizone tree on the CONTROL brain (mirrors what section 3 did, but
#    divisive + gated + transferable instead of one flat Leiden)
# clust = Clustering(embedding_control)              # wraps the Harmonized embedding + raw lipids
# root_node, clustering_log = learn_euclid_clustering(
#     clust,
#     K=60,                # over-segment each branch into 60 micro-clusters before re-aggregating to 2
#     min_voxels=150,      # do not split a group smaller than 150 pixels
#     min_diff_lipids=2,   # accept a split only if >=2 lipids differ (Mann-Whitney + BH)
#     min_fc=0.2, pthr=0.05,
#     ACCTHR=0.6,          # the per-node XGBoost must reach 0.6 test accuracy to keep the split
#     max_depth=5,         # cap the tree depth
#     combinations=20,     # how many top-factor combinations to try per split
#     do_plotting=True,
# )
# # root_node is the tree of trained per-node XGBoost classifiers; clustering_log holds level_1..level_N

# 4) TRANSFER the learned tree onto the FULL data (control + pregnant) by walking root->leaf
#    (mirrors section 5's kNN vote, but as an XGBoost tree-descent that is exact and hierarchical)
# clust_full = Clustering(embedding_full)
# paths_df = apply_euclid_clustering(root_node, clust_full.adatamaia,
#                                    clust_full.standardized_embeddings_GLOBAL)
# # paths_df gives every pixel its lipizone path; paint_lipizones() then colors the maps.

print("illustrative cell: see the prose above; we ran the transparent equivalent in sections 3-5.")

💡 **HINT.** Notice the parallel, line for line. `learn_euclid_clustering` is the careful cousin of
your `sc.tl.leiden` on control; `apply_euclid_clustering` is the careful cousin of your
`KNeighborsClassifier` transfer onto pregnant. Same two movements, learn on the reference and transfer to
the query, just with a divisive tree and per-node classifiers instead of a flat cut and a neighbour vote.
You now understand the package well enough to read its source.

❓ **QUESTION.** EUCLID's splitter recomputes a *local* NMF inside each branch before deciding how to
split it. Our flat Leiden used the *same* 16 global factors everywhere. What might a local NMF reveal
inside, say, the cortex, that a single global 16-factor embedding of the whole brain would smooth away?
Hold that thought: building the divisive splitter, local NMF and all, is exactly the next notebook.

## what you did

You turned a 16-dimensional lipid embedding into named brain territories and carried those names across
conditions. Concretely:

- You built a **kNN graph** by hand and saw it is just a table of who-is-near-whom in lipid space, then
  let **Leiden** cut that graph into communities, the **lipizones**, with the **resolution** knob setting
  how fine they are.
- You painted the lipizones on the brain (spatially coherent territories), laid them beside the Allen
  reference, and measured agreement with the **Adjusted Rand Index**. The ARI near 0.05, the same as the
  published lipizones, is the lesson: lipizones partially track anatomy but are their own organizing
  principle, because membrane lipid composition reflects biology that classical anatomy does not capture.
- You ran **kNN label transfer** across a **Harmony**-shared space (harmonized on section only, to keep
  the pregnancy biology), copying the control lipizones onto the pregnant brain by majority vote, with a
  free **confidence** map that lit up the uncertain boundaries.
- You read the **production path**: EUCLID's divisive tree with per-node XGBoost classifiers does a more
  careful version of the same learn-then-transfer logic, which you will rebuild from scratch next.
